# Notebook 7: Answers to the reviewers

This notebook answers the reviewers' requests that need new experiments. It **re-uses your finished runs** (the `ext_*_seed*.csv` files from Notebook 6) and only adds what is new.

| Reviewer request | Where it is answered |
|---|---|
| Compare against confident learning (cleanlab) | Results 3 (competitors) and the new `cl_*` features |
| Report the real time to compute the meta-features, and compare it with training | Results 1 |
| More seeds (3 became 10) | all results; the old seeds 0-2 are re-used, seeds 3-9 are new |
| What does the method give beyond knowing the noise level? | Results 4 (ranking datasets at a fixed noise level, and a "which datasets to check first" test) |
| Stronger baselines and more evidence | Results 3, 5, 6 and 7 (different meta-models, leave-one-dataset-out, leave-one-cluster-out, a reversed train/test split) |
| A more realistic noise type | Part D (instance-dependent noise) |

**Models:** logistic regression, random forest, XGBoost and LightGBM. The neural network (MLP) is dropped from the new seeds because it never gave a gain and is the slowest model.

**Running time:** the new seeds take the longest (they train 4 models on 23 settings for 48 datasets, 7 more times). Everything is saved per dataset and seed, so you can **stop and run the cell again later**. The results use every seed that is finished for all datasets, so you can also stop early.

**Before the real run:** set `FAST_TEST = True` once (a few minutes) to check that all cells work, then set it back to `False`.

In [ ]:
!pip install -q openml lightgbm cleanlab

## Setup: where results are saved

`OLD_DIR` is the folder where Notebook 6 saved its files. If it does not exist, seeds 0-2 are simply computed again (slower, but the result is the same).

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/data_quality_paper_v7'
    OLD_DIR = '/content/drive/MyDrive/data_quality_paper_extended'
except ImportError:
    OUTPUT_DIR = 'results_v7'
    OLD_DIR = 'results_extended'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Saving results to:', OUTPUT_DIR)
print('Old Notebook 6 files found:', len([f for f in os.listdir(OLD_DIR) if f.startswith('ext_')]) if os.path.isdir(OLD_DIR) else 0)

In [ ]:
import time, glob, warnings
import numpy as np
import pandas as pd
import openml

warnings.filterwarnings('ignore')

FAST_TEST = False                 # True = only 4 small datasets, to test that everything works
OLD_SEEDS = [0, 1, 2]             # seeds of Notebook 6 (re-used)
NEW_SEEDS = [3, 4, 5, 6, 7, 8, 9] # new seeds (you can shorten this list to save time)
MODELS = ['logreg', 'rf', 'xgb', 'lgbm']
TREE_MODELS = ['rf', 'xgb', 'lgbm']
METRICS = ['acc', 'bacc', 'f1']   # accuracy, balanced accuracy, macro-F1
METRIC_NAMES = {'acc': 'accuracy', 'bacc': 'balanced accuracy', 'f1': 'macro-F1'}
FILE_PREFIX = 'n7'

ORIGINAL_IDS = [3, 15, 29, 31, 37, 38, 44, 50, 1049, 1067, 1480,
                11, 23, 46, 54, 307, 12, 14, 16, 22, 28]
NEW_IDS = [18, 182, 188, 458, 469, 1050, 1063, 1068, 1462, 1475, 1487, 1489, 1494, 1497,
           1510, 1464, 4538, 6332, 23381, 40499, 40701, 40966, 40975, 40982, 40983, 40984, 40994]
DATASET_IDS = ORIGINAL_IDS + NEW_IDS
if FAST_TEST:
    ORIGINAL_IDS = ORIGINAL_IDS[:2]
    NEW_IDS = [1063, 1464]
    DATASET_IDS = ORIGINAL_IDS + NEW_IDS
    OLD_SEEDS = [0]
    NEW_SEEDS = [1, 2]
ALL_SEEDS = OLD_SEEDS + NEW_SEEDS


def load_dataset(did):
    ds = openml.datasets.get_dataset(did)
    X, y, _, _ = ds.get_data(target=ds.default_target_attribute)
    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            codes = X[col].astype('category').cat.codes.astype(float)
            X[col] = codes.where(codes >= 0, np.nan)
    X = X.astype(float).reset_index(drop=True)
    y = pd.factorize(y)[0]
    return X, y, ds.name

## Part A: data problems and models

Unchanged from Notebook 6 (the same defects, levels, settings and models), so the old result files fit exactly.

In [ ]:
def inject_missing(X, level, seed=0):
    rng = np.random.default_rng(seed)
    return X.mask(rng.random(X.shape) < level)


def inject_label_noise(y, level, seed=0):
    rng = np.random.default_rng(seed)
    y = y.copy()
    classes = np.unique(y)
    idx = rng.choice(len(y), int(level * len(y)), replace=False)
    for i in idx:
        y[i] = rng.choice(classes[classes != y[i]])
    return y


def inject_class_noise(y, level, seed=0):
    # class-dependent noise: a label moves to the next class (cyclic)
    rng = np.random.default_rng(seed + 12345)
    y = y.copy()
    classes = np.unique(y)
    pos = {c: i for i, c in enumerate(classes)}
    idx = rng.choice(len(y), int(level * len(y)), replace=False)
    for i in idx:
        y[i] = classes[(pos[y[i]] + 1) % len(classes)]
    return y


def inject_duplicates(X, y, level, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), int(level * len(X)), replace=True)
    return pd.concat([X, X.iloc[idx]], ignore_index=True), np.concatenate([y, y[idx]])


def inject_imbalance(X, y, level, seed=0):
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    biggest = classes[np.argmax(counts)]
    keep = []
    for c in classes:
        idx = np.where(y == c)[0]
        if c != biggest:
            idx = rng.choice(idx, max(2, int(round(len(idx) * (1 - level)))), replace=False)
        keep.extend(idx)
    keep = np.sort(np.array(keep))
    return X.iloc[keep].reset_index(drop=True), y[keep]


def apply_defect(X_tr, y_tr, defect, level, seed=0):
    if defect == 'missing':
        return inject_missing(X_tr, level, seed), y_tr
    if defect == 'label_noise':
        return X_tr, inject_label_noise(y_tr, level, seed)
    if defect == 'class_noise':
        return X_tr, inject_class_noise(y_tr, level, seed)
    if defect == 'duplicates':
        return inject_duplicates(X_tr, y_tr, level, seed)
    if defect == 'imbalance':
        return inject_imbalance(X_tr, y_tr, level, seed)
    raise ValueError('unknown defect: ' + defect)


DEFECTS = ['missing', 'label_noise', 'class_noise', 'duplicates', 'imbalance']
DEFECT_ORDER = ['label_noise', 'class_noise', 'imbalance', 'duplicates', 'missing']
COMBINED = ['missing', 'label_noise', 'duplicates', 'imbalance']   # as in the original paper
LEVELS = [0.05, 0.1, 0.2, 0.4]
COMBO_LEVELS = [0.1, 0.2]


def apply_defects(X, y, setting, seed=0):
    for d in DEFECT_ORDER:
        if d in setting:
            X, y = apply_defect(X, y, d, setting[d], seed)
    return X, y


def build_settings():
    settings = [{}]
    for d in DEFECTS:
        for lv in LEVELS:
            settings.append({d: lv})
    for lv in COMBO_LEVELS:
        settings.append({d: lv for d in COMBINED})
    return settings


SETTINGS = build_settings()
print('Number of settings per dataset:', len(SETTINGS))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


def make_model(name):
    imputer = SimpleImputer(strategy='median')
    if name == 'logreg':
        return make_pipeline(imputer, StandardScaler(), LogisticRegression(max_iter=1000))
    if name == 'rf':
        return make_pipeline(imputer, RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0))
    if name == 'xgb':
        return make_pipeline(imputer, XGBClassifier(n_estimators=100, n_jobs=-1, random_state=0, verbosity=0))
    if name == 'lgbm':
        return make_pipeline(imputer, LGBMClassifier(n_estimators=100, n_jobs=-1, random_state=0, verbose=-1))
    if name == 'mlp':
        return make_pipeline(imputer, StandardScaler(), MLPClassifier(
            hidden_layer_sizes=(64,), max_iter=200, early_stopping=True, random_state=0))
    raise ValueError('unknown model: ' + name)


def fit_predict(name, X_fit, y_fit, X_eval):
    # trains a model and predicts; the neural network falls back to a version without
    # early stopping if the internal validation split fails (tiny classes)
    try:
        return make_model(name).fit(X_fit, y_fit).predict(X_eval)
    except Exception:
        if name != 'mlp':
            raise
        model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), MLPClassifier(
            hidden_layer_sizes=(64,), max_iter=200, early_stopping=False, random_state=0))
        return model.fit(X_fit, y_fit).predict(X_eval)


def scores(y_true, y_pred):
    return {'acc': accuracy_score(y_true, y_pred),
            'bacc': balanced_accuracy_score(y_true, y_pred),
            'f1': f1_score(y_true, y_pred, average='macro')}

## Part B: meta-features, now with timing and cleanlab features

The 16 old meta-features are unchanged. **New:**

* **Timing:** how long the meta-features take (`t_meta`, of which `t_landmark` is the three cheap learners), compared with training the target model on the clean data (`t_fit`).
* **Four confident-learning features** (`cl_*`), computed with the `cleanlab` library on out-of-sample predictions of a small random forest (50 trees, 3 folds, at most 1000 rows): the share of examples cleanlab flags as label issues (`cl_issue_rate`), cleanlab's estimated noise rate (`cl_noise_rate`), the average probability given to the given label (`cl_self_conf`) and the share of examples where the forest disagrees with the given label (`cl_disagree`). The time for them is `t_cl`.

Note on the "before training" wording: the landmarking learners **are** trained, but they are tiny (a stump, a depth-3 tree, naive Bayes) and use at most 1000 rows, so the cost does not grow with dataset size. What is avoided is training the **target model** on the full damaged data. Results 1 reports the real numbers.

In [ ]:
from scipy.stats import entropy
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

META_COLS = ['n_rows', 'n_features', 'n_classes', 'rows_per_feature',
             'missing_rate', 'duplicate_rate', 'imbalance_ratio', 'outlier_rate',
             'mean_skew', 'mean_kurtosis', 'mean_abs_corr',
             'class_entropy', 'mean_mutual_info',
             'lm_stump', 'lm_tree3', 'lm_nb']


def compute_meta_features(X, y, seed=0):
    rng = np.random.default_rng(seed)
    n, p = X.shape
    classes, counts = np.unique(y, return_counts=True)
    prop = counts / counts.sum()
    mf = {'n_rows': n, 'n_features': p, 'n_classes': len(classes), 'rows_per_feature': n / p,
          'missing_rate': float(X.isna().to_numpy().mean()),
          'duplicate_rate': float(X.assign(_y=y).duplicated().mean()),
          'imbalance_ratio': float(counts.min() / counts.max()),
          'class_entropy': float(entropy(prop) / np.log(len(classes))) if len(classes) > 1 else 0.0}
    A = X.to_numpy(dtype=float)
    mu = np.nanmean(A, axis=0)
    sd = np.nanstd(A, axis=0)
    sd[sd == 0] = np.nan
    z = np.abs((A - mu) / sd)
    valid = ~np.isnan(z)
    mf['outlier_rate'] = float((z[valid] > 3).mean()) if valid.any() else 0.0
    mf['mean_skew'] = float(X.skew().mean())
    mf['mean_kurtosis'] = float(X.kurt().mean())
    if p > 1:
        C = X.corr().abs().to_numpy()
        mf['mean_abs_corr'] = float(np.nanmean(C[np.triu_indices(p, k=1)]))
    else:
        mf['mean_abs_corr'] = 0.0
    m = min(n, 1000)
    idx = rng.choice(n, m, replace=False)
    t_lm = time.perf_counter()
    Xs = X.iloc[idx]
    ys = y[idx]
    Xi = Xs.fillna(Xs.median()).fillna(0.0).to_numpy()
    cols = rng.choice(p, min(p, 20), replace=False)
    try:
        mf['mean_mutual_info'] = float(mutual_info_classif(Xi[:, cols], ys, random_state=seed).mean())
    except Exception:
        mf['mean_mutual_info'] = np.nan

    def landmark(model):
        try:
            return float(cross_val_score(model, Xi, ys, cv=3).mean())
        except Exception:
            return np.nan

    mf['lm_stump'] = landmark(DecisionTreeClassifier(max_depth=1, random_state=0))
    mf['lm_tree3'] = landmark(DecisionTreeClassifier(max_depth=3, random_state=0))
    mf['lm_nb'] = landmark(GaussianNB())
    mf['t_landmark'] = time.perf_counter() - t_lm
    return mf

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from cleanlab.filter import find_label_issues
from cleanlab.count import compute_confident_joint, estimate_joint

CL_COLS = ['cl_issue_rate', 'cl_noise_rate', 'cl_self_conf', 'cl_disagree']


def cl_pred_probs(Xi, ys, K, seed):
    # out-of-sample class probabilities from a small random forest (3 folds)
    n = len(ys)
    probs = np.zeros((n, K))
    counts = np.bincount(ys, minlength=K)
    folds = StratifiedKFold(3, shuffle=True, random_state=seed) if counts.min() >= 3 else KFold(3, shuffle=True, random_state=seed)
    for tr, te in folds.split(Xi, ys):
        clf = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=seed).fit(Xi[tr], ys[tr])
        probs[np.ix_(te, clf.classes_)] = clf.predict_proba(Xi[te])
    return probs


def compute_cl_features(X, y, seed=0):
    out = {c: np.nan for c in CL_COLS}
    rng = np.random.default_rng(seed + 99)
    n = len(y)
    m = min(n, 1000)
    idx = rng.choice(n, m, replace=False)
    Xs = X.iloc[idx]
    Xi = Xs.fillna(Xs.median()).fillna(0.0).to_numpy()
    classes, ys = np.unique(np.asarray(y)[idx], return_inverse=True)
    K = len(classes)
    if K < 2:
        return out
    try:
        P = cl_pred_probs(Xi, ys, K, seed)
        out['cl_self_conf'] = float(P[np.arange(m), ys].mean())
        out['cl_disagree'] = float((P.argmax(axis=1) != ys).mean())
        P = P / np.clip(P.sum(axis=1, keepdims=True), 1e-12, None)
        out['cl_issue_rate'] = float(np.mean(find_label_issues(ys, P)))
        cj = compute_confident_joint(ys, P)
        out['cl_noise_rate'] = float(1.0 - np.trace(estimate_joint(ys, P, confident_joint=cj)))
    except Exception:
        pass
    return out


def all_meta_features(X, y, seed=0):
    t0 = time.perf_counter()
    mf = compute_meta_features(X, y, seed)
    mf['t_meta'] = time.perf_counter() - t0
    t1 = time.perf_counter()
    mf.update(compute_cl_features(X, y, seed))
    mf['t_cl'] = time.perf_counter() - t1
    return mf

## Part A (continued): run the experiments

One file is saved per dataset and seed. For seeds 0-2 the **models are not trained again**: the notebook reads the finished degradation values from your Notebook 6 files, rebuilds the same damaged training sets (the random draws depend only on the seed), checks that they match, and computes the new meta-features and timings. For seeds 3-9 everything is computed. If you stop the notebook, run the cell again and it continues.

In [ ]:
def old_path(did, seed):
    return os.path.join(OLD_DIR, 'ext_' + str(did) + '_seed' + str(seed) + '.csv')


def new_path(did, seed):
    return os.path.join(OUTPUT_DIR, FILE_PREFIX + '_' + str(did) + '_seed' + str(seed) + '.csv')


def fit_times(X_tr, y_tr, X_te, y_te):
    # trains every target model on the clean data; returns scores and training times
    sc, tm = {}, {}
    for m in MODELS:
        t0 = time.perf_counter()
        pred = fit_predict(m, X_tr, y_tr, X_te)
        tm[m] = time.perf_counter() - t0
        sc[m] = scores(y_te, pred)
    return sc, tm


def run_dataset_seed(did, seed, X, y, name):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
    X_tr = X_tr.reset_index(drop=True)
    clean, tfit = fit_times(X_tr, y_tr, X_te, y_te)
    old = None
    if seed in OLD_SEEDS and os.path.exists(old_path(did, seed)):
        old = pd.read_csv(old_path(did, seed))
        old = old[old['model'].isin(MODELS)]
    rows = []
    mismatch = False
    for sid, setting in enumerate(SETTINGS):
        Xd, yd = apply_defects(X_tr, y_tr, setting, seed)
        mf = all_meta_features(Xd, yd, seed)
        if old is not None:
            o = old[old['setting_id'] == sid]
            if len(o) == 0 or abs(o['missing_rate'].iloc[0] - mf['missing_rate']) > 1e-6 or abs(o['n_rows'].iloc[0] - mf['n_rows']) > 0.5:
                mismatch = True
                break
        for m in MODELS:
            if old is not None:
                om = o[o['model'] == m]
                if len(om) == 0:
                    mismatch = True
                    break
                degs = {k: float(om['deg_' + k].iloc[0]) for k in METRICS}
            else:
                sc = clean[m] if not setting else scores(y_te, fit_predict(m, Xd, yd, X_te))
                degs = {k: clean[m][k] - sc[k] for k in METRICS}
            row = {'dataset': name, 'dataset_id': did, 'seed': seed, 'setting_id': sid, 'model': m, 't_fit': tfit[m]}
            for d in DEFECTS:
                row['inj_' + d] = setting.get(d, 0.0)
            row.update(mf)
            for k in METRICS:
                row['deg_' + k] = degs[k]
            rows.append(row)
        if mismatch:
            break
    if mismatch:
        print('  old file for', did, 'seed', seed, 'does not match the rebuilt data; computing the models again')
        rows = []
        for sid, setting in enumerate(SETTINGS):
            Xd, yd = apply_defects(X_tr, y_tr, setting, seed)
            mf = all_meta_features(Xd, yd, seed)
            for m in MODELS:
                sc = clean[m] if not setting else scores(y_te, fit_predict(m, Xd, yd, X_te))
                row = {'dataset': name, 'dataset_id': did, 'seed': seed, 'setting_id': sid, 'model': m, 't_fit': tfit[m]}
                for d in DEFECTS:
                    row['inj_' + d] = setting.get(d, 0.0)
                row.update(mf)
                for k in METRICS:
                    row['deg_' + k] = clean[m][k] - sc[k]
                rows.append(row)
    return pd.DataFrame(rows)


failed = []
for did in DATASET_IDS:
    todo = [s for s in ALL_SEEDS if not os.path.exists(new_path(did, s))]
    if not todo:
        continue
    try:
        X, y, name = load_dataset(did)
    except Exception as e:
        print('FAILED to load dataset', did, ':', repr(e)[:200])
        failed.append(did)
        continue
    for seed in todo:
        t0 = time.time()
        try:
            df = run_dataset_seed(did, seed, X, y, name)
        except Exception as e:
            print('FAILED dataset', did, 'seed', seed, ':', repr(e)[:200])
            failed.append(did)
            break
        df.to_csv(new_path(did, seed), index=False)
        print(f'done {did} ({name}), seed {seed}, in {(time.time() - t0) / 60:.1f} min')
print('Failed datasets:', sorted(set(failed)))

## Part C: prepare the results

A dataset is used if it finished seeds 0-2. Then every seed that is finished for **all** these datasets is used (printed below), seeds are averaged, and clean rows are removed.

In [ ]:
files = sorted(glob.glob(os.path.join(OUTPUT_DIR, FILE_PREFIX + '_*_seed*.csv')))
all_df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
all_df = all_df[all_df['dataset_id'].isin(DATASET_IDS) & all_df['seed'].isin(ALL_SEEDS)]
have = all_df.groupby('dataset_id')['seed'].apply(lambda s: set(s.unique()))
good_ids = [d for d, s in have.items() if set(OLD_SEEDS) <= s]
all_df = all_df[all_df['dataset_id'].isin(good_ids)]
SEEDS_USED = [s for s in ALL_SEEDS if all(s in have[d] for d in good_ids)]
all_df = all_df[all_df['seed'].isin(SEEDS_USED)]
print('Datasets:', len(good_ids), '| Seeds used:', SEEDS_USED)

ALLF = META_COLS + CL_COLS
agg_cols = ALLF + ['deg_' + k for k in METRICS] + ['inj_' + d for d in DEFECTS]
data = all_df.groupby(['dataset', 'dataset_id', 'setting_id', 'model'], as_index=False)[agg_cols].mean()
data = data[data['setting_id'] != 0].reset_index(drop=True)


def defect_type(row):
    active = [d for d in DEFECTS if row['inj_' + d] > 0]
    return active[0] if len(active) == 1 else 'combined'


data['defect_type'] = data.apply(defect_type, axis=1)
data['level'] = [r['inj_' + r['defect_type']] if r['defect_type'] != 'combined' else r['inj_missing']
                 for _, r in data.iterrows()]
ORIG = [d for d in ORIGINAL_IDS if d in good_ids]
print('Rows used for scoring:', len(data), '| Round-1 datasets:', len(ORIG), '| Round-2 datasets:', len(good_ids) - len(ORIG))

## Results 1: how long do the meta-features take? (reviewers 2 and 4)

Compared with training each target model on the **clean** training set of the same dataset and seed. `all_features` = all 16 meta-features plus the 4 cleanlab features; `landmark_only` = the three landmarking learners, which are the costly part of the paper's 7-feature meta-model (the other 4 features are simple counts). The ratio is feature time divided by training time; `share_faster` is the share of dataset-seed pairs where computing the features is faster than training that model. All times are seconds on this machine; report the machine type (Colab CPU) in the paper.

In [ ]:
tm = all_df[all_df['setting_id'] > 0].groupby(['dataset_id', 'seed'], as_index=False)[['t_meta', 't_landmark', 't_cl']].mean()
fit = all_df[all_df['setting_id'] == 0].groupby(['dataset_id', 'seed', 'model'], as_index=False)['t_fit'].mean()
tt = fit.merge(tm, on=['dataset_id', 'seed'])
tt['t_features'] = tt['t_meta'] + tt['t_cl']
tt['ratio'] = tt['t_features'] / tt['t_fit']
tt['ratio_landmark'] = tt['t_landmark'] / tt['t_fit']
rows = []
for m in MODELS:
    d = tt[tt['model'] == m]
    rows.append({'model': m, 'median_t_landmark_s': d['t_landmark'].median(), 'median_t_meta_s': d['t_meta'].median(),
                 'median_t_cleanlab_s': d['t_cl'].median(), 'median_t_features_total_s': d['t_features'].median(),
                 'median_t_train_s': d['t_fit'].median(), 'median_ratio_all_features': d['ratio'].median(),
                 'share_faster_all_features': float((d['ratio'] < 1).mean()),
                 'median_ratio_landmark_only': d['ratio_landmark'].median(),
                 'share_faster_landmark_only': float((d['ratio_landmark'] < 1).mean())})
timing = pd.DataFrame(rows).round(3)
timing.to_csv(os.path.join(OUTPUT_DIR, 'result1_timing.csv'), index=False)
print('The target-model training time is for one fit on the clean training set; the meta-features are computed once and serve every model.')
timing

## Meta-models and helper functions

`predict_method` builds every meta-model that is compared:

* `rf_small`: the paper's meta-model (random forest, 7 features)
* `rf_small_cl`: the same plus the 4 cleanlab features
* `rf_lm_cl`: only the 3 landmarking features and the 4 cleanlab features
* `rf_all`: random forest on all 20 features
* `ridge_all`: ridge regression on all 20 features
* `gb_all`: gradient boosting on all 20 features
* `cl_only`, `cl_issues_only`: **cleanlab alone**: the loss is predicted from cleanlab's estimated noise rate (or flagged-issue share) with a monotone (isotonic) fit. This is the "natural competitor" the reviewer asked for.
* `rf_small_told`: `rf_small` plus the true noise level (an unrealistic upper bound)

Baselines that are not meta-models: `avg_drop` (the average loss in training) and `told_noise` (the average loss seen in training at the true noise level).

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr, wilcoxon

SMALL = ['missing_rate', 'duplicate_rate', 'imbalance_ratio', 'class_entropy', 'lm_stump', 'lm_tree3', 'lm_nb']
LM = ['lm_stump', 'lm_tree3', 'lm_nb']


def per_ds(y_true, y_pred, ids):
    return pd.Series(np.abs(np.asarray(y_true) - np.asarray(y_pred))).groupby(np.asarray(ids)).mean()


def wil(a, b, alt='less'):
    try:
        return wilcoxon(a, b, alternative=alt).pvalue
    except Exception:
        return np.nan


def holm(pvals):
    # Holm step-down correction; returns adjusted p-values in the original order
    p = np.asarray(pvals, dtype=float)
    out = np.full(len(p), np.nan)
    ok = ~np.isnan(p)
    idx = np.where(ok)[0][np.argsort(p[ok])]
    m = len(idx)
    run = 0.0
    for rank, i in enumerate(idx):
        run = max(run, (m - rank) * p[i])
        out[i] = min(1.0, run)
    return out


def boot_ci(err_meta, err_base, n=2000, seed=0):
    # 95% bootstrap interval (resampling datasets) of the relative error reduction of the meta-model
    rng = np.random.default_rng(seed)
    a, b = np.asarray(err_meta), np.asarray(err_base)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a))
        vals.append(1 - a[i].mean() / b[i].mean())
    return np.percentile(vals, [2.5, 97.5])
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.cluster import KMeans

SMALL_CL = SMALL + CL_COLS
LM_CL = LM + CL_COLS
METHODS = {
    'rf_small': ('rf', SMALL),
    'rf_small_cl': ('rf', SMALL_CL),
    'rf_lm_cl': ('rf', LM_CL),
    'rf_all': ('rf', ALLF),
    'ridge_all': ('ridge', ALLF),
    'gb_all': ('gb', ALLF),
    'cl_only': ('iso', ['cl_noise_rate']),
    'cl_issues_only': ('iso', ['cl_issue_rate']),
    'rf_small_told': ('rf', SMALL + ['inj_label_noise']),
}


def make_meta(kind='rf'):
    imp = SimpleImputer(strategy='median', keep_empty_features=True)
    if kind == 'ridge':
        return make_pipeline(imp, StandardScaler(), Ridge(alpha=1.0))
    if kind == 'gb':
        return make_pipeline(imp, GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.05,
                                                            subsample=0.8, random_state=0))
    return make_pipeline(imp, RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.5,
                                                    n_jobs=-1, random_state=0))


def predict_method(name, tr, te, y_tr):
    kind, cols = METHODS[name]
    if kind == 'iso':
        x_tr = tr[cols[0]].to_numpy(dtype=float)
        x_te = te[cols[0]].to_numpy(dtype=float)
        fill = np.nanmedian(x_tr) if np.isfinite(x_tr).any() else 0.0
        x_tr = np.where(np.isnan(x_tr), fill, x_tr)
        x_te = np.where(np.isnan(x_te), fill, x_te)
        return IsotonicRegression(out_of_bounds='clip').fit(x_tr, y_tr).predict(x_te)
    return make_meta(kind).fit(tr[cols], y_tr).predict(te[cols])


def told_noise_pred(tr, te, k):
    trl = tr[tr['defect_type'] == 'label_noise']
    tab = trl.groupby('level')['deg_' + k].mean()
    return np.array([tab.iloc[np.abs(tab.index.to_numpy() - l).argmin()] for l in te['level']])


def planned_split(m):
    tr = data[(data['model'] == m) & data['dataset_id'].isin(ORIG) & (data['defect_type'] != 'class_noise')]
    te = data[(data['model'] == m) & ~data['dataset_id'].isin(ORIG) & (data['defect_type'] == 'label_noise')].reset_index(drop=True)
    return tr, te


def summarize(y_te, ids, preds, base='avg_drop'):
    # per-method error, error reduction against the baseline with a bootstrap interval, wins and p-value
    e = {n: per_ds(y_te, p, ids) for n, p in preds.items()}
    out = []
    for n, p in preds.items():
        row = {'method': n, 'MAE': float(np.abs(y_te - p).mean())}
        if n != base:
            lo, hi = boot_ci(e[n], e[base])
            row.update({'reduction_%': 100 * (1 - e[n].mean() / e[base].mean()), 'CI95_low_%': 100 * lo, 'CI95_high_%': 100 * hi,
                        'better_on': str(int((e[n] < e[base]).sum())) + '/' + str(len(e[n])),
                        'p_one_sided': wil(e[n], e[base]), 'rank_corr': spearmanr(y_te, p)[0]})
        out.append(row)
    return out

## Results 2: the planned test again, with more seeds (reviewer 2, point 5)

Same planned test as in the paper (train on the Round-1 datasets, test on the label-noise cases of the Round-2 datasets), for the paper's meta-model, now with the averaged targets of all finished seeds. Holm correction is over all rows of this table. Compare the width of the intervals with the old table (Notebook 6, Results 3).

In [ ]:
rows = []
store = {}
for k in METRICS:
    for m in MODELS:
        tr, te = planned_split(m)
        y_tr, y_te, ids = tr['deg_' + k].to_numpy(), te['deg_' + k].to_numpy(), te['dataset_id'].to_numpy()
        preds = {'avg_drop': np.full(len(te), y_tr.mean()), 'told_noise': told_noise_pred(tr, te, k),
                 'rf_small': predict_method('rf_small', tr, te, y_tr)}
        store[(k, m)] = (te, preds)
        for r in summarize(y_te, ids, {'avg_drop': preds['avg_drop'], 'rf_small': preds['rf_small'], 'told_noise': preds['told_noise']}):
            if r['method'] == 'rf_small':
                r.update({'measure': METRIC_NAMES[k], 'model': m})
                r['MAE_avg_drop'] = float(np.abs(y_te - preds['avg_drop']).mean())
                r['MAE_told_noise'] = float(np.abs(y_te - preds['told_noise']).mean())
                rows.append(r)
res2 = pd.DataFrame(rows)
res2['p_holm'] = holm(res2['p_one_sided'].to_numpy())
res2 = res2[['measure', 'model', 'MAE_avg_drop', 'MAE', 'MAE_told_noise', 'reduction_%', 'CI95_low_%', 'CI95_high_%',
             'better_on', 'p_one_sided', 'p_holm', 'rank_corr']].rename(columns={'MAE': 'MAE_meta'}).round(4)
res2.to_csv(os.path.join(OUTPUT_DIR, 'result2_planned_test_more_seeds.csv'), index=False)
res2

## Results 3: competitors, including cleanlab alone (reviewer 2, point 1; reviewer 4: more features)

Planned test, label noise, for accuracy and macro-F1. `reduction_%` is against the average-drop baseline. Rows `cl_only` and `cl_issues_only` are the cleanlab-only competitors. `rf_small_cl` shows whether adding the cleanlab features to the paper's meta-model helps. Holm correction is over all rows of the table.

In [ ]:
rows = []
for k in ['acc', 'f1']:
    for m in MODELS:
        tr, te = planned_split(m)
        y_tr, y_te, ids = tr['deg_' + k].to_numpy(), te['deg_' + k].to_numpy(), te['dataset_id'].to_numpy()
        preds = {'avg_drop': np.full(len(te), y_tr.mean()), 'told_noise': told_noise_pred(tr, te, k)}
        for name in METHODS:
            preds[name] = predict_method(name, tr, te, y_tr)
        for r in summarize(y_te, ids, preds):
            if r['method'] != 'avg_drop':
                r.update({'measure': METRIC_NAMES[k], 'model': m})
                rows.append(r)
res3 = pd.DataFrame(rows)
res3['p_holm'] = holm(res3['p_one_sided'].to_numpy())
res3 = res3[['measure', 'model', 'method', 'MAE', 'reduction_%', 'CI95_low_%', 'CI95_high_%', 'better_on', 'p_one_sided', 'p_holm', 'rank_corr']].round(4)
res3.to_csv(os.path.join(OUTPUT_DIR, 'result3_competitors.csv'), index=False)
pd.set_option('display.width', 200)
res3.pivot_table(index=['measure', 'model'], columns='method', values='reduction_%').round(1)

## Results 4: what does the meta-model give beyond the noise level? (reviewer 2, point 2)

If the noise level is known, the "told the noise level" baseline predicts the same loss for every dataset. A useful meta-model should also tell **which datasets suffer more** at the same noise level.

Two measures, on the label-noise cases of the Round-2 datasets, at each noise level separately:

* `within_level_corr`: the rank correlation between the predicted and the real loss across datasets at one noise level, averaged over levels. For the "told the noise level" baseline it is zero by definition, because it gives every dataset the same number.
* **Triage test:** if only the top 25% of the datasets (those with the highest predicted loss) can be checked for label errors, what share of the total real loss is in them? Random choice gives 25%; a perfect ranking gives the `oracle` share. The bootstrap interval is for the difference to random choice (resampling datasets).

In [ ]:
def triage_share(y, score, frac=0.25):
    k = max(1, int(round(frac * len(y))))
    top = np.argsort(-score)[:k]
    return y[top].sum() / y.sum() if y.sum() > 0 else np.nan


def triage_ci(y, score, frac=0.25, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        s = triage_share(y[i], score[i], frac)
        if not np.isnan(s):
            vals.append(s - max(1, int(round(frac * len(i)))) / len(i))
    return np.percentile(vals, [2.5, 97.5])


rows = []
for m in MODELS:
    tr, te = planned_split(m)
    y_tr = tr['deg_acc'].to_numpy()
    preds = {name: predict_method(name, tr, te, y_tr) for name in ['rf_small', 'rf_small_cl', 'cl_only']}
    for name, p in preds.items():
        wl, tri, tri_lo, tri_hi, orc = [], [], [], [], []
        for lv in sorted(te['level'].unique()):
            if lv < 0.1:
                continue
            msk = (te['level'] == lv).to_numpy()
            if msk.sum() < 6:
                continue
            y = te['deg_acc'].to_numpy()[msk]
            wl.append(spearmanr(y, p[msk])[0])
            tri.append(triage_share(y, p[msk]))
            lo, hi = triage_ci(y, p[msk])
            tri_lo.append(lo)
            tri_hi.append(hi)
            orc.append(triage_share(y, y))
        rows.append({'model': m, 'method': name, 'within_level_corr': np.nanmean(wl), 'share_of_loss_in_top25%': np.nanmean(tri),
                     'diff_vs_random_CI_low': np.nanmean(tri_lo), 'diff_vs_random_CI_high': np.nanmean(tri_hi),
                     'oracle_share': np.nanmean(orc), 'random_share': 0.25})
res4 = pd.DataFrame(rows).round(3)
res4.to_csv(os.path.join(OUTPUT_DIR, 'result4_beyond_noise_level.csv'), index=False)
res4

## Results 5: leave-one-dataset-out on all 48 datasets (more seeds, with and without cleanlab features)

All damaged settings, accuracy. `reduction_%` against the average-drop baseline, with a bootstrap interval; Holm correction over the table.

In [ ]:
rows = []
for m in MODELS:
    d = data[data['model'] == m].reset_index(drop=True)
    y = d['deg_acc'].to_numpy()
    groups = d['dataset_id'].to_numpy()
    preds = {'avg_drop': np.zeros(len(d)), 'rf_small': np.zeros(len(d)), 'rf_small_cl': np.zeros(len(d))}
    for tr_i, te_i in LeaveOneGroupOut().split(d, y, groups):
        preds['avg_drop'][te_i] = y[tr_i].mean()
        for name in ['rf_small', 'rf_small_cl']:
            preds[name][te_i] = predict_method(name, d.iloc[tr_i], d.iloc[te_i], y[tr_i])
    for r in summarize(y, groups, preds):
        if r['method'] != 'avg_drop':
            r['model'] = m
            rows.append(r)
res5 = pd.DataFrame(rows)
res5['p_holm'] = holm(res5['p_one_sided'].to_numpy())
res5 = res5[['model', 'method', 'MAE', 'reduction_%', 'CI95_low_%', 'CI95_high_%', 'better_on', 'p_one_sided', 'p_holm', 'rank_corr']].round(4)
res5.to_csv(os.path.join(OUTPUT_DIR, 'result5_leave_one_dataset_out.csv'), index=False)
res5

## Results 6: leave-one-cluster-out and the reversed split (reviewer 2, point 5: more evidence)

* **Leave-one-cluster-out:** the datasets are grouped into 6 clusters by size, number of features and number of classes (k-means); each cluster is held out in turn. This is stricter than leaving out one dataset, because similar datasets leave together.
* **Reversed split:** train on the 27 Round-2 datasets and test on the label-noise cases of the 21 Round-1 datasets. The meta-model now has more training datasets and is tested on different ones.

In [ ]:
rows = []
ds_info = data.groupby('dataset_id')[['n_rows', 'n_features', 'n_classes']].median()
Z = np.column_stack([np.log(ds_info['n_rows']), np.log(ds_info['n_features']), ds_info['n_classes']])
sd = Z.std(axis=0)
sd[sd == 0] = 1.0
Z = (Z - Z.mean(axis=0)) / sd
nclust = min(6, max(2, len(ds_info) // 4))
cl_id = pd.Series(KMeans(nclust, n_init=10, random_state=0).fit_predict(Z), index=ds_info.index)
print('Cluster sizes:', cl_id.value_counts().sort_index().tolist())
for m in MODELS:
    d = data[data['model'] == m].reset_index(drop=True)
    y = d['deg_acc'].to_numpy()
    groups = d['dataset_id'].to_numpy()
    cg = d['dataset_id'].map(cl_id).to_numpy()
    preds = {'avg_drop': np.zeros(len(d)), 'rf_small': np.zeros(len(d)), 'rf_small_cl': np.zeros(len(d))}
    for tr_i, te_i in LeaveOneGroupOut().split(d, y, cg):
        preds['avg_drop'][te_i] = y[tr_i].mean()
        for name in ['rf_small', 'rf_small_cl']:
            preds[name][te_i] = predict_method(name, d.iloc[tr_i], d.iloc[te_i], y[tr_i])
    for r in summarize(y, groups, preds):
        if r['method'] != 'avg_drop':
            r.update({'model': m, 'test': 'leave-one-cluster-out (all damaged settings)'})
            rows.append(r)
for m in MODELS:
    tr = data[(data['model'] == m) & ~data['dataset_id'].isin(ORIG) & (data['defect_type'] != 'class_noise')]
    te = data[(data['model'] == m) & data['dataset_id'].isin(ORIG) & (data['defect_type'] == 'label_noise')].reset_index(drop=True)
    if te['dataset_id'].nunique() < 2 or tr['dataset_id'].nunique() < 2:
        continue
    y_tr, y_te, ids = tr['deg_acc'].to_numpy(), te['deg_acc'].to_numpy(), te['dataset_id'].to_numpy()
    preds = {'avg_drop': np.full(len(te), y_tr.mean())}
    for name in ['rf_small', 'rf_small_cl']:
        preds[name] = predict_method(name, tr, te, y_tr)
    for r in summarize(y_te, ids, preds):
        if r['method'] != 'avg_drop':
            r.update({'model': m, 'test': 'reversed split (label noise)'})
            rows.append(r)
res6 = pd.DataFrame(rows)
res6['p_holm'] = holm(res6['p_one_sided'].to_numpy())
res6 = res6[['test', 'model', 'method', 'MAE', 'reduction_%', 'CI95_low_%', 'CI95_high_%', 'better_on', 'p_one_sided', 'p_holm', 'rank_corr']].round(4)
res6.to_csv(os.path.join(OUTPUT_DIR, 'result6_cluster_out_and_reversed.csv'), index=False)
res6

## Part D: a more realistic noise type: instance-dependent label noise

In real data, labels are wrong mostly for examples that are hard or ambiguous. Here a random forest first estimates how uncertain each training example is (out-of-bag probabilities). Examples are chosen for flipping **with probability proportional to their uncertainty**, and each chosen label is moved to the class the forest finds most likely besides the true one. The share of flipped labels is 5, 10, 20 or 40%.

This is still synthetic, but it depends on the features, so it is much harder than random noise. The paper should call it "instance-dependent noise", **not** "real noise". It runs on the Round-2 datasets with seeds 0-2.

In [ ]:
def inject_instance_noise(X, y, level, seed=0):
    # label noise that depends on the example: uncertain examples are flipped more often
    rng = np.random.default_rng(seed + 777)
    y = np.asarray(y)
    Xi = X.fillna(X.median()).fillna(0.0).to_numpy()
    rf = RandomForestClassifier(n_estimators=100, oob_score=True, n_jobs=-1, random_state=seed).fit(Xi, y)
    proba = rf.oob_decision_function_
    proba = np.where(np.isnan(proba), 1.0 / proba.shape[1], proba)
    classes = rf.classes_
    pos = {c: i for i, c in enumerate(classes)}
    srt = np.sort(proba, axis=1)
    margin = srt[:, -1] - srt[:, -2] if proba.shape[1] > 1 else np.ones(len(y))
    weight = (1.0 - margin) + 1e-3
    n_flip = int(level * len(y))
    idx = rng.choice(len(y), n_flip, replace=False, p=weight / weight.sum())
    y2 = y.copy()
    for i in idx:
        pr = proba[i].copy()
        pr[pos[y[i]]] = -1.0
        y2[i] = classes[int(np.argmax(pr))]
    return y2


def run_instance_noise(did, seeds=(0,), models=MODELS):
    X, y, name = load_dataset(did)
    rows = []
    for seed in seeds:
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
        X_tr = X_tr.reset_index(drop=True)
        clean = {m: scores(y_te, fit_predict(m, X_tr, y_tr, X_te)) for m in models}
        for lv in LEVELS:
            yd = inject_instance_noise(X_tr, y_tr, lv, seed)
            mf = all_meta_features(X_tr, yd, seed)
            for m in models:
                sc = scores(y_te, fit_predict(m, X_tr, yd, X_te))
                row = {'dataset': name, 'dataset_id': did, 'seed': seed, 'level': lv, 'model': m}
                row.update(mf)
                for k in METRICS:
                    row['deg_' + k] = clean[m][k] - sc[k]
                rows.append(row)
    return pd.DataFrame(rows)


inst_files = []
for did in NEW_IDS:
    for seed in OLD_SEEDS[:3] if not FAST_TEST else OLD_SEEDS + NEW_SEEDS[:1]:
        path = os.path.join(OUTPUT_DIR, 'inst7_' + str(did) + '_seed' + str(seed) + '.csv')
        if not os.path.exists(path):
            t0 = time.time()
            try:
                df = run_instance_noise(did, seeds=[seed])
            except Exception as e:
                print('FAILED dataset', did, ':', repr(e)[:200])
                continue
            df.to_csv(path, index=False)
            print(f'done {did} ({df["dataset"].iloc[0]}), seed {seed}, in {(time.time() - t0) / 60:.1f} min')
        inst_files.append(path)
print('Finished instance-noise files:', len(inst_files))

In [ ]:
inst = pd.concat([pd.read_csv(f) for f in inst_files if os.path.exists(f)], ignore_index=True)
inst = inst.groupby(['dataset', 'dataset_id', 'level', 'model'], as_index=False)[ALLF + ['deg_' + k for k in METRICS]].mean()
inst['inj_label_noise'] = inst['level']
inst['defect_type'] = 'instance_noise'

rows = []
for k in ['acc', 'f1']:
    for m in MODELS:
        tr = data[(data['model'] == m) & data['dataset_id'].isin(ORIG) & (data['defect_type'] != 'class_noise')]
        te = inst[inst['model'] == m].reset_index(drop=True)
        if te['dataset_id'].nunique() < 2:
            continue
        y_tr, y_te, ids = tr['deg_' + k].to_numpy(), te['deg_' + k].to_numpy(), te['dataset_id'].to_numpy()
        preds = {'avg_drop': np.full(len(te), y_tr.mean()), 'told_noise': told_noise_pred(tr, te, k)}
        for name in ['rf_small', 'rf_small_cl', 'cl_only']:
            preds[name] = predict_method(name, tr, te, y_tr)
        for r in summarize(y_te, ids, preds):
            if r['method'] != 'avg_drop':
                r.update({'measure': METRIC_NAMES[k], 'model': m, 'n_datasets': len(np.unique(ids)), 'real_mean_loss': float(y_te.mean())})
                rows.append(r)
res7 = pd.DataFrame(rows)
if len(res7):
    res7['p_holm'] = holm(res7['p_one_sided'].to_numpy())
    res7 = res7[['measure', 'model', 'method', 'n_datasets', 'real_mean_loss', 'MAE', 'reduction_%', 'CI95_low_%', 'CI95_high_%',
                 'better_on', 'p_one_sided', 'p_holm', 'rank_corr']].round(4)
    res7.to_csv(os.path.join(OUTPUT_DIR, 'result7_instance_dependent_noise.csv'), index=False)
res7

## Done

Send back the seven result files from the Drive folder `data_quality_paper_v7` (`result1_timing.csv` ... `result7_instance_dependent_noise.csv`), or paste the tables, and the paper will be updated with them.

Also send the line printed in Part C (`Datasets: ... | Seeds used: ...`).